## Building Nonsequential Models Using Custom Modules

In [1]:
import torch
import torch.nn as nn

In [2]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = fetch_california_housing()
df = data.data
target = data.target

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=(device == "cuda"))

In [4]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU()
        )
        self.output_layer = nn.Linear(40 + n_features, 1)
    
    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [5]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [6]:
torch.manual_seed(42)
n_features = X_train.shape[1]
learning_rate = 0.002

model = WideAndDeep(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
n_epochs = 20

In [7]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 1.3319
Epoch 2/20, Loss: 0.6117
Epoch 3/20, Loss: 0.5662
Epoch 4/20, Loss: 0.5345
Epoch 5/20, Loss: 0.5104
Epoch 6/20, Loss: 0.4929
Epoch 7/20, Loss: 0.4804
Epoch 8/20, Loss: 0.4683
Epoch 9/20, Loss: 0.4589
Epoch 10/20, Loss: 0.4501
Epoch 11/20, Loss: 0.4423
Epoch 12/20, Loss: 0.4354
Epoch 13/20, Loss: 0.4282
Epoch 14/20, Loss: 0.4228
Epoch 15/20, Loss: 0.4167
Epoch 16/20, Loss: 0.4116
Epoch 17/20, Loss: 0.4075
Epoch 18/20, Loss: 0.4018
Epoch 19/20, Loss: 0.3971
Epoch 20/20, Loss: 0.3934


In [8]:
X_test = X_test.to(device)
y_test = y_test.to(device)

with torch.no_grad():
    y_test_pred = model(X_test)

mse(y_test, y_test_pred)

tensor(0.4076, device='cuda:0')

What if you want to send a subset of the features through the wide path and a different subset through the deep path, best approach is to let the model take two separate tensors.

### Building Models with Multiple Inputs

In [9]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_wide, n_deep):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_deep, 50), nn.ReLU(),
            nn.Linear(50, 30), nn.ReLU()
        )
        self.output_layer = nn.Linear(30 + n_wide, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [10]:
X_train_wide, X_train_deep = X_train[:, :5], X_train[:, 2:]
X_test_wide, X_test_deep = X_test[:, :5], X_test[:, 2:]
X_valid_wide, X_valid_deep = X_valid[:, :5], X_valid[:, 2:]

train_data_wd = TensorDataset(X_train_wide, X_train_deep, y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)

In [11]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch_wide, X_batch_deep, y_batch in train_loader:
            X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device),  X_batch_deep.to(device), y_batch.to(device)
            y_pred = model(X_batch_wide, X_batch_deep)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [12]:
torch.manual_seed(42)
learning_rate = 0.002

model = WideAndDeepV3(n_wide=5, n_deep=6).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
n_epochs = 20

In [13]:
train(model, optimizer, mse, train_loader_wd, n_epochs)

Epoch 1/20, Loss: 1.4707
Epoch 2/20, Loss: 0.6274
Epoch 3/20, Loss: 0.5645
Epoch 4/20, Loss: 0.5280
Epoch 5/20, Loss: 0.5056
Epoch 6/20, Loss: 0.4905
Epoch 7/20, Loss: 0.4799
Epoch 8/20, Loss: 0.4714
Epoch 9/20, Loss: 0.4641
Epoch 10/20, Loss: 0.4579
Epoch 11/20, Loss: 0.4521
Epoch 12/20, Loss: 0.4473
Epoch 13/20, Loss: 0.4431
Epoch 14/20, Loss: 0.4393
Epoch 15/20, Loss: 0.4360
Epoch 16/20, Loss: 0.4328
Epoch 17/20, Loss: 0.4301
Epoch 18/20, Loss: 0.4273
Epoch 19/20, Loss: 0.4247
Epoch 20/20, Loss: 0.4226


In [14]:
X_test_wide, X_test_deep = X_test_wide.to(device), X_test_deep.to(device)
y_test = y_test.to(device)

with torch.no_grad():
    y_test_pred = model(X_test_wide, X_test_deep)

mse(y_test, y_test_pred)

tensor(0.4373, device='cuda:0')

### Building Models with Multiple Outputs

Let’s add an auxiliary output to our Wide & Deep model to ensure the deep part can make good predictions on its own.

In [15]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_wide, n_deep):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_deep, 50), nn.ReLU(),
            nn.Linear(50, 30), nn.ReLU()
        )
        self.output_layer = nn.Linear(n_wide + 30, 1)
        self.aux_output_layer = nn.Linear(30, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)

        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        return main_output, aux_output

In [16]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch_wide, X_batch_deep, y_batch in train_loader:
            X_batch_wide, X_batch_deep, y_batch = X_batch_wide.to(device),  X_batch_deep.to(device), y_batch.to(device)
            y_pred, y_pred_aux = model(X_batch_wide, X_batch_deep)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [17]:
torch.manual_seed(42)
learning_rate = 0.002

model = WideAndDeepV4(n_wide=5, n_deep=6).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
n_epochs = 20

In [18]:
train(model, optimizer, mse, train_loader_wd, n_epochs)

Epoch 1/20, Loss: 2.0101
Epoch 2/20, Loss: 0.7883
Epoch 3/20, Loss: 0.7056
Epoch 4/20, Loss: 0.6597
Epoch 5/20, Loss: 0.6249
Epoch 6/20, Loss: 0.5975
Epoch 7/20, Loss: 0.5748
Epoch 8/20, Loss: 0.5561
Epoch 9/20, Loss: 0.5404
Epoch 10/20, Loss: 0.5268
Epoch 11/20, Loss: 0.5148
Epoch 12/20, Loss: 0.5050
Epoch 13/20, Loss: 0.4958
Epoch 14/20, Loss: 0.4875
Epoch 15/20, Loss: 0.4807
Epoch 16/20, Loss: 0.4748
Epoch 17/20, Loss: 0.4699
Epoch 18/20, Loss: 0.4650
Epoch 19/20, Loss: 0.4613
Epoch 20/20, Loss: 0.4581


In [19]:
X_test_wide, X_test_deep = X_test_wide.to(device), X_test_deep.to(device)
y_test = y_test.to(device)

with torch.no_grad():
    y_pred_main, y_pred_aux  = model(X_test_wide, X_test_deep)

mse(y_test, y_pred_main)

tensor(0.4346, device='cuda:0')

## Building an Image Classifier with PyTorch

### Using TorchVision to Load the Dataset 

In [22]:
import torchvision 
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)

test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000]
)

100%|██████████| 26.4M/26.4M [01:59<00:00, 222kB/s] 
100%|██████████| 29.5k/29.5k [00:00<00:00, 44.2kB/s]
100%|██████████| 4.42M/4.42M [00:11<00:00, 390kB/s] 
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.52MB/s]


In [23]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [28]:
X_sample, y_sample = train_data[0]
X_sample.shape

torch.Size([1, 28, 28])

In [26]:
X_sample.dtype

torch.float32

In [31]:
train_and_valid_data.classes[y_sample]

'Ankle boot'